In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import joblib



In [2]:
# Regenerate features with new additions
from src.features.engineer import build_features
from src.data.ingest import build_universe, download_prices, clean_prices



cache = Path("../data/raw/prices.parquet")
tickers = build_universe()
raw_df = download_prices(tickers, cache)
clean_df, _ = clean_prices(raw_df)

features_df = build_features(clean_df)
features_clean = features_df.dropna()
features_clean.to_parquet("../data/processed/features.parquet")
print(features_clean.shape)


Loading cached data from ../data/raw/prices.parquet
Dropping 7 tickers: ['AAF.L', 'CCEP.L', 'EDV.L', 'HLN.L', 'MNG.L', 'MTLN.L', 'PSH.L']
(246078, 9)


In [4]:
# Load features
features_clean = pd.read_parquet("../data/processed/features.parquet")

# Define feature columns and targets
FEATURE_COLS = [
    "momentum", "volatility_21d", "drawdown_52w",
    "relative_strength", "volume_ratio_20", "beta_252", "vix"
]


# Three-way split
# Tuning window: used for CV-based hyperparameter search
# Held-out: touched exactly once for final evaluation

tuning = features_clean[
    (features_clean.index.get_level_values("date") >= "2019-01-01") &
    (features_clean.index.get_level_values("date") <  "2024-01-01")
]

held_out = features_clean[
    features_clean.index.get_level_values("date") >= "2024-01-01"
]

print(f"Tuning window: {tuning.shape}")
print(f"Held-out:      {held_out.shape}")

X_tune     = tuning[FEATURE_COLS]
y_tune_ret = tuning["forward_return"]
y_tune_vol = tuning["forward_volatility"]

X_held     = held_out[FEATURE_COLS]
y_held_ret = held_out["forward_return"]
y_held_vol = held_out["forward_volatility"]


Tuning window: (119691, 9)
Held-out:      (54312, 9)


The held-out set (2024-01-01 onwards) is reserved for final evaluation only and is not used for model selection, feature engineering decisions, or any intermediate analysis. It is evaluated exactly once.


In [5]:
class PurgedTimeSeriesSplit:
    """
    TimeSeriesSplit with purging and embargo to prevent target-window leakage
    in panel data. Operates on date level, purging all tickers simultaneously.

    For each validation fold [val_start, val_end]:
      - Any training row whose 21-day target window overlaps
        [val_start - embargo, val_end] is removed from training.

    """

    def __init__(self, n_splits=5, horizon_td=21, embargo_td=5):
        self.n_splits   = n_splits
        self.horizon_td = horizon_td  # prediction horizon in trading days
        self.embargo_td = embargo_td  # embargo buffer in trading days

    def split(self, X, y=None, groups=None):
        dates        = X.index.get_level_values("date")
        unique_dates = np.sort(dates.unique())

        tscv = TimeSeriesSplit(n_splits=self.n_splits)

        for train_date_idx, val_date_idx in tscv.split(unique_dates):
            train_dates = unique_dates[train_date_idx]
            val_dates   = unique_dates[val_date_idx]

            val_start_pos = np.searchsorted(unique_dates, val_dates.min())

            # Purge training rows whose target window reaches into the
            # embargo zone before val_start
            purge_pos = max(0, val_start_pos - self.horizon_td - self.embargo_td)
            purge_cutoff = unique_dates[purge_pos]

            purged_train_dates = train_dates[train_dates < purge_cutoff]

            train_mask = dates.isin(set(purged_train_dates))
            val_mask   = dates.isin(set(val_dates))

            yield np.where(train_mask)[0], np.where(val_mask)[0]

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


purged_cv = PurgedTimeSeriesSplit(n_splits=5, horizon_td=21, embargo_td=5)
print("Purged CV splitter ready.")

# Sanity check: print fold sizes
for i, (tr, val) in enumerate(purged_cv.split(X_tune)):
    print(f"Fold {i+1}: train={len(tr):,}  val={len(val):,}")

Purged CV splitter ready.
Fold 1: train=17,763  val=19,902
Fold 2: train=37,665  val=19,902
Fold 3: train=57,567  val=19,902
Fold 4: train=77,469  val=19,902
Fold 5: train=97,371  val=19,902


In [6]:
# --- Return baseline: rolling historical mean (pooled across tickers) ---
# On each date, predict the mean of all forward returns observed before that date

all_dates = features_clean.index.get_level_values("date").unique().sort_values()

# Compute expanding mean of forward_return up to each date
daily_mean_return = (
    features_clean["forward_return"]
    .groupby(level="date").mean()
    .expanding().mean()
    .shift(1)  # use only past information
)

# Align to held-out set
held_out_dates = held_out.index.get_level_values("date")
baseline_ret_pred = daily_mean_return.reindex(held_out_dates).values

# --- Volatility baseline: persistence (current vol_21d as forecast) ---
baseline_vol_pred = held_out["volatility_21d"].values

# --- Baseline metrics ---
from sklearn.metrics import mean_absolute_error

r2_base_ret   = r2_score(y_held_ret, baseline_ret_pred)
rmse_base_ret = np.sqrt(mean_squared_error(y_held_ret, baseline_ret_pred))
mae_base_ret  = mean_absolute_error(y_held_ret, baseline_ret_pred)

r2_base_vol   = r2_score(y_held_vol, baseline_vol_pred)
rmse_base_vol = np.sqrt(mean_squared_error(y_held_vol, baseline_vol_pred))
mae_base_vol  = mean_absolute_error(y_held_vol, baseline_vol_pred)

print("Return baseline (historical mean):")
print(f"  R²={r2_base_ret:.4f}  RMSE={rmse_base_ret:.4f}  MAE={mae_base_ret:.4f}")
print("\nVolatility baseline (persistence):")
print(f"  R²={r2_base_vol:.4f}  RMSE={rmse_base_vol:.4f}  MAE={mae_base_vol:.4f}")


Return baseline (historical mean):
  R²=-0.0036  RMSE=0.0796  MAE=0.0590

Volatility baseline (persistence):
  R²=-0.3661  RMSE=0.1369  MAE=0.0901


In [ ]:
param_grid = {
    "max_depth":        [5, 10, 15, None],
    "min_samples_leaf": [50, 100, 200]
}

rf_tuned = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Return RF
gs_ret = GridSearchCV(rf_tuned, param_grid, cv=purged_cv,
                      scoring="r2", verbose=1)
gs_ret.fit(X_tune, y_tune_ret)
print(f"Return RF  — best params: {gs_ret.best_params_}, CV R²: {gs_ret.best_score_:.4f}")


Fitting 5 folds for each of 12 candidates, totalling 60 fits
